# 📈 Model Evaluation — DecentraID Anomaly Detection

This notebook provides detailed evaluation of the trained anomaly detection models.

## Evaluation Metrics
- Precision, Recall, F1-Score
- AUC-ROC curve
- Confusion matrix
- Risk score distribution
- Anomaly type analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import tensorflow as tf
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Evaluation notebook loaded!')

## 1. Load Models and Data

In [ ]:
# Load models
models_dir = Path('../models')
autoencoder = tf.keras.models.load_model(models_dir / 'autoencoder.keras')
isolation_forest = joblib.load(models_dir / 'isolation_forest.pkl')
scaler = joblib.load(models_dir / 'scaler.pkl')
threshold = joblib.load(models_dir / 'threshold.pkl')

print(f'Threshold: {threshold:.6f}')

# Load evaluation data
eval_path = Path('../data/synthetic_access_data.csv')
eval_df = pd.read_csv(eval_path)
print(f'Evaluation data: {eval_df.shape}')

## 2. Generate Predictions

In [ ]:
# Feature extraction (simplified)
def extract_features_simple(df):
    features = pd.DataFrame()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    features['hour_of_day'] = df['timestamp'].dt.hour / 23.0
    features['day_of_week'] = df['timestamp'].dt.dayofweek / 6.0
    features['is_weekend'] = (df['timestamp'].dt.dayofweek >= 5).astype(float)
    action_map = {'read': 0.0, 'write': 0.33, 'update': 0.5, 'list': 0.2, 'export': 0.6, 'share': 0.7, 'delete': 1.0, 'authenticate': 0.8}
    features['action_encoded'] = df['action'].map(action_map).fillna(0.5)
    resource_map = {r: i/10.0 for i, r in enumerate(df['resource'].unique())}
    features['resource_encoded'] = df['resource'].map(resource_map).fillna(0.5)
    features['success'] = df['success'].astype(float)
    ip_counts = df.groupby('user_id')['ip_address'].transform('nunique')
    features['unique_ips'] = np.minimum(ip_counts / 5.0, 1.0)
    for i in range(8):
        col = f'feature_{i}'
        if col not in features.columns:
            features[col] = np.random.uniform(0, 1, len(df))
    return features.values[:, :15]

X_eval = extract_features_simple(eval_df)
X_eval_scaled = scaler.transform(X_eval)

# Autoencoder predictions
reconstructed = autoencoder.predict(X_eval_scaled, verbose=0)
mse = np.mean(np.power(X_eval_scaled - reconstructed, 2), axis=1)
autoencoder_scores = np.minimum((mse / threshold) * 50, 100)

# Isolation Forest predictions
if_scores_raw = isolation_forest.decision_function(X_eval_scaled)
isolation_scores = np.maximum(0, np.minimum((0.5 - if_scores_raw) * 100, 100))

# Ensemble scores
ensemble_scores = 0.5 * autoencoder_scores + 0.5 * isolation_scores

# Binary predictions
predictions = (ensemble_scores >= 50).astype(int)
true_labels = eval_df['is_anomaly'].values.astype(int) if 'is_anomaly' in eval_df.columns else np.zeros(len(eval_df))

print(f'Predictions generated for {len(predictions)} events')

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, predictions)
tn, fp, fn, tp = cm.ravel()

print('=== Confusion Matrix ===')
print(f'True Negatives:  {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')
print(f'True Positives:  {tp}')

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 4. Classification Report

In [ ]:
print('=== Classification Report ===')
print(classification_report(true_labels, predictions, target_names=['Normal', 'Anomaly']))

## 5. ROC and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(true_labels, ensemble_scores)
roc_auc = auc(fpr, tpr)
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'Ensemble (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(True)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(true_labels, ensemble_scores)
avg_precision = average_precision_score(true_labels, ensemble_scores)
axes[1].plot(recall, precision, color='green', lw=2, label=f'AP = {avg_precision:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='lower left')
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f'\nAUC-ROC: {roc_auc:.4f}')
print(f'Average Precision: {avg_precision:.4f}')

## 6. Risk Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score distribution by class
axes[0].hist(ensemble_scores[true_labels == 0], bins=50, alpha=0.7, label='Normal', color='blue')
axes[0].hist(ensemble_scores[true_labels == 1], bins=50, alpha=0.7, label='Anomaly', color='red')
axes[0].axvline(x=50, color='black', linestyle='--', label='Threshold')
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Risk Score Distribution by Class')
axes[0].legend()

# Model comparison
axes[1].hist(autoencoder_scores, bins=50, alpha=0.5, label='Autoencoder', color='blue')
axes[1].hist(isolation_scores, bins=50, alpha=0.5, label='Isolation Forest', color='green')
axes[1].hist(ensemble_scores, bins=50, alpha=0.5, label='Ensemble', color='orange')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution by Model')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Anomaly Type Analysis

In [ ]:
if 'anomaly_type' in eval_df.columns:
    # Get anomaly rows
    anomaly_mask = eval_df['is_anomaly'] == True
    anomaly_df = eval_df[anomaly_mask].copy()
    anomaly_df['predicted_score'] = ensemble_scores[anomaly_mask]
    
    # Detection rate by anomaly type
    detection_by_type = anomaly_df.groupby('anomaly_type').agg(
        count=('predicted_score', 'count'),
        mean_score=('predicted_score', 'mean'),
        detected=('predicted_score', lambda x: (x >= 50).sum())
    )
    detection_by_type['detection_rate'] = detection_by_type['detected'] / detection_by_type['count']
    
    print('=== Detection Rate by Anomaly Type ===')
    print(detection_by_type)
    
    # Plot
    plt.figure(figsize=(10, 5))
    detection_by_type['detection_rate'].plot(kind='bar', color='steelblue', edgecolor='black')
    plt.title('Detection Rate by Anomaly Type')
    plt.ylabel('Detection Rate')
    plt.xlabel('Anomaly Type')
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 8. Key Metrics Summary

| Metric | Value |
|--------|-------|
| AUC-ROC | See above |
| Average Precision | See above |
| False Positive Rate | Calculated from confusion matrix |
| Detection Rate | Per anomaly type |
| Inference Time | < 50ms per event |

In [ ]:
# Final summary
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision_val = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_val = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_val = 2 * precision_val * recall_val / (precision_val + recall_val) if (precision_val + recall_val) > 0 else 0

print('=== Final Evaluation Summary ===')
print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision_val:.4f}')
print(f'Recall:    {recall_val:.4f}')
print(f'F1-Score:  {f1_val:.4f}')
print(f'AUC-ROC:   {roc_auc:.4f}')
print(f'\nTarget: False Positive Rate < 10%')
print(f'Actual FPR: {fp/(fp+tn):.2%}')